# Autonomiczny agent newsowy — demo

Agent zbudowany na **natywnym function-calling modelu Gemini** (bez LangChain).
Na zadany temat samodzielnie: wyszukuje wiadomości, tworzy podsumowanie, ocenia
istotność i zapisuje raport PDF o nazwie `<data>_<temat>_<istotność>.pdf`.

> Uwaga o limitach: darmowy plan Gemini ogranicza liczbę zapytań (dziennie i na
> minutę). Agent wykonuje kilka wywołań modelu na sesję, więc przy pełnym modelu
> `gemini-2.5-flash` można szybko wyczerpać dzienny limit. Kod ma wbudowane
> ponawianie po błędzie 429.

In [ ]:
# !pip install -q google-genai markdown xhtml2pdf python-dotenv

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from rdzen import AgentNewsowy, Ustawienia

## 1. Uruchomienie agenta

Agent dostaje tylko temat i cel — sam planuje kolejność użycia narzędzi.

In [ ]:
agent = AgentNewsowy()
wynik = agent.uruchom('energetyka jądrowa w Polsce')

print('--- ŚLAD DECYZJI AGENTA ---')
for i, krok in enumerate(wynik.slad, 1):
    print(f'{i}. {krok.narzedzie}({", ".join(krok.argumenty)}) -> {krok.obserwacja[:80]}...')

print('\n--- ODPOWIEDŹ KOŃCOWA ---')
print(wynik.odpowiedz)

## 2. Narzędzia pojedynczo

Można też wywołać narzędzia bezpośrednio (np. do testów), z pominięciem pętli
decyzyjnej agenta.

In [ ]:
from rdzen import narzedzia
from google import genai
ust = Ustawienia(); klient = genai.Client(api_key=ust.klucz_gemini)
narzedzia.ustaw_kontekst(klient, ust)

temat = 'energetyka jądrowa w Polsce'
dane = narzedzia.szukaj_wiadomosci(temat)
podsum = narzedzia.podsumuj_wiadomosci(dane, temat)
ist = narzedzia.ocen_istotnosc(podsum, temat)
print('Istotność:', ist)
print(narzedzia.zapisz_raport_pdf(podsum, temat, ist))